# SentinelFlow — 00: ROCm + vLLM + Qwen Setup

**Stack:** AMD ROCm · vLLM · Qwen2.5-VL (or Qwen2-VL)  
**Purpose:** Install dependencies, verify GPU, launch vLLM server, confirm Qwen responds.

> Run this notebook **once** before any other notebook in the suite.

## 1. System check — ROCm GPU visibility

In [ ]:
import subprocess, sys

# ROCm GPU info
result = subprocess.run(['rocm-smi', '--showproductname', '--showmeminfo', 'vram'],
                        capture_output=True, text=True)
print(result.stdout or result.stderr)

# PyTorch ROCm check
import torch
print(f'PyTorch version  : {torch.__version__}')
print(f'ROCm available   : {torch.cuda.is_available()}')
print(f'GPU count        : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    total_gb = props.total_memory / 1024**3
    print(f'  GPU {i}: {props.name}  {total_gb:.1f} GB VRAM')

## 2. Install dependencies

In [ ]:
# Install vLLM ROCm build + supporting libraries
# Use the ROCm-specific wheel — do NOT install the CUDA build
!pip install vllm --extra-index-url https://download.pytorch.org/whl/rocm6.1 -q
!pip install openai httpx pyyaml numpy opencv-python-headless \
             pillow langgraph langchain-core tiktoken \
             accelerate transformers -q
print('Installation complete')

## 3. Configuration — edit these values

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
QWEN_MODEL        = 'Qwen/Qwen2.5-VL-7B-Instruct'  # or Qwen2-VL-7B-Instruct
VLLM_HOST         = '0.0.0.0'
VLLM_PORT         = 8000
VLLM_GPU_UTIL     = 0.90        # fraction of VRAM to use
VLLM_MAX_MODEL_LEN= 4096        # reduce if VRAM is tight
VLLM_DTYPE        = 'float16'   # 'bfloat16' if your GPU supports it
VLLM_TENSOR_PARALLEL = 1        # set to number of GPUs for multi-GPU

# Derived endpoint used by all other notebooks
VLLM_BASE_URL = f'http://localhost:{VLLM_PORT}/v1'

print(f'Model          : {QWEN_MODEL}')
print(f'vLLM endpoint  : {VLLM_BASE_URL}')

## 4. Save config for all notebooks

In [ ]:
import yaml, pathlib

config = {
    'qwen_model'        : QWEN_MODEL,
    'vllm_host'         : VLLM_HOST,
    'vllm_port'         : VLLM_PORT,
    'vllm_base_url'     : VLLM_BASE_URL,
    'vllm_gpu_util'     : VLLM_GPU_UTIL,
    'vllm_max_model_len': VLLM_MAX_MODEL_LEN,
    'vllm_dtype'        : VLLM_DTYPE,
    'vllm_tensor_parallel': VLLM_TENSOR_PARALLEL,
}

pathlib.Path('config.yml').write_text(yaml.dump(config))
print('Saved config.yml')
print(yaml.dump(config))

## 5. Launch vLLM server (background process)

This starts the OpenAI-compatible server. It stays running while other notebooks use it.  
The cell streams logs — **wait for `Application startup complete`** before proceeding.

In [ ]:
import subprocess, os, time, threading

env = os.environ.copy()
# ROCm-specific env vars
env['HIP_VISIBLE_DEVICES']     = '0'          # comma-separated GPU indices
env['ROCR_VISIBLE_DEVICES']    = '0'
env['HSA_OVERRIDE_GFX_VERSION']= '11.0.0'    # MI300X=9.4.2, RX7900=11.0.0
                                               # check with: rocminfo | grep 'gfx'

cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model',                 QWEN_MODEL,
    '--host',                  VLLM_HOST,
    '--port',                  str(VLLM_PORT),
    '--dtype',                 VLLM_DTYPE,
    '--gpu-memory-utilization',str(VLLM_GPU_UTIL),
    '--max-model-len',         str(VLLM_MAX_MODEL_LEN),
    '--tensor-parallel-size',  str(VLLM_TENSOR_PARALLEL),
    '--trust-remote-code',
    '--served-model-name',     'qwen',
]

print('Starting vLLM server...')
print(' '.join(cmd))
print('─' * 60)

proc = subprocess.Popen(cmd, env=env,
                        stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT,
                        text=True, bufsize=1)

def stream_logs():
    for line in proc.stdout:
        print(line, end='', flush=True)
        if 'Application startup complete' in line:
            print('\n✓ vLLM server is ready')

t = threading.Thread(target=stream_logs, daemon=True)
t.start()

# Keep cell alive for 90s max — enough for model load
t.join(timeout=90)
print('(Log streaming ended — server continues running)')

## 6. Health check + Qwen smoke test

In [ ]:
import time, httpx
from openai import OpenAI

# Wait for server
for attempt in range(30):
    try:
        r = httpx.get(f'http://localhost:{VLLM_PORT}/health', timeout=3)
        if r.status_code == 200:
            print(f'✓ Server healthy after {attempt+1} attempts')
            break
    except Exception:
        pass
    time.sleep(3)
else:
    raise RuntimeError('vLLM server did not become healthy in 90s')

# OpenAI-compatible client pointing at local vLLM
client = OpenAI(
    base_url=VLLM_BASE_URL,
    api_key='not-needed',     # vLLM doesn't require a real key
)

# List available models
models = client.models.list()
print('Available models:', [m.id for m in models.data])

# Text smoke test
resp = client.chat.completions.create(
    model='qwen',
    messages=[{'role': 'user',
               'content': 'Reply with exactly 3 words: the architecture name of your model family.'}],
    max_tokens=16,
    temperature=0.0,
)
print('Qwen text reply:', resp.choices[0].message.content)
print('\n✓ Setup complete — proceed to 01_edge_filter.ipynb')

## 7. GPU memory snapshot after load

In [ ]:
result = subprocess.run(['rocm-smi', '--showmeminfo', 'vram'], capture_output=True, text=True)
print(result.stdout)